# 4. The compiled program vs trace simulation

If the host CPU is not relevant part of the evaluation, we could use trace simulation to speedup the simulation time.
This considers that control logic built close to the memory controller issues commands to DRAM module. 
The mapping pass can also write a **trace**: the DRAM commands the schedule would issue, grouped by when they may
start. gem5's trace player replayed it directly against the memory controllers.
That measures the memory alone, as if commands arrived for free.

The compiled path issues the same commands from a real processor. 
Comparing the two tell us whether *the program issue exactly what the cost model estimated* and *how much CPU issuing costs*

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # tutorial/, for _infra
from _infra import pud

CASES = [("vadd_1048576", 32), ("vadd_1048576", 64), ("relu_1048576", 32), ("relu_1048576", 64)]
MEM = "ddr"

rows = []
for name, banks in CASES:
    k = pud.compile_kernel(name, banks, MEM, trace=True)
    c = pud.simulate(k.exe, MEM)
    t = pud.replay(k.mapping.trace, MEM)
    rows.append((name, banks, k, c, t))
    print(f"{name} at {banks} banks done")

In [ ]:
print(f"{'kernel':14} {'banks':>5} {'ops compiled':>12} {'ops trace':>10} {'':>5} "
      f"{'compiled us':>11} {'trace us':>9} {'ratio':>6}")
for name, banks, _, c, t in rows:
    same = "=" if c.row_ops == t.row_ops else "DIFF"
    print(f"{name:14} {banks:5} {c.row_ops:12} {t.row_ops:10} {same:>5} "
          f"{c.makespan_us:11.3f} {t.makespan_us:9.3f} {c.makespan_us / t.makespan_us:6.3f}")

**The counts are equal.** What the ILP estimated, what the host CPU issued and what the
trace expands to are the same commands. Over the full benchmark set (14 kernels ×
2 memories × 5 bank counts) this holds in all 140 configurations.

**The makespans are close within one channel and apart beyond it.** In one channel
the memory is the bottleneck, and the compiled program keeps it busy. With two
channels the memory can accept commands twice as fast, but one core at 2 GHz still
needs a few cycles per command. The difference is the cost of issuing the schedule,
which the trace cannot express.

## Is it really the core?

If it is, a faster core should close the gap. Run the two-channel `vadd` again at
4 GHz:

In [ ]:
name, banks, k, c2, t = rows[1]
fast = pud.simulate(k.exe, MEM, cpu_clock="4GHz")
print(f"2 GHz: {c2.makespan_us:8.3f} us  ({c2.makespan_us / t.makespan_us:.3f} x the trace)")
print(f"4 GHz: {fast.makespan_us:8.3f} us  ({fast.makespan_us / t.makespan_us:.3f} x the trace)")

The memory did not change, only the core, so what moved is the host-side cost.
Reducing it is compiler work: fewer instructions per command, and code that stays
in the instruction cache.